# 02 - Modelo Baseline

El baseline utiliza la imagen completa de la palta.

Arquitectura:

**Imagen completa ? MobileNetV3Small ? Clasificaci?n RUSSET / PICADURA**

Este modelo servir? como referencia para comparar posteriormente los modelos por c?mara y el modelo multivista.


In [ ]:
from pathlib import Path
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score
)

SEMILLA = 42

random.seed(SEMILLA)
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)

RUTA_DATASET = Path(r"..\DATASET_PALTA")

RUTA_TRAIN = RUTA_DATASET / "train"
RUTA_VALIDACION = RUTA_DATASET / "validation"
RUTA_TEST = RUTA_DATASET / "test"

ALTO = 256
ANCHO = 512

TAMANO_LOTE = 8

EXTENSIONES = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

print("TensorFlow:", tf.__version__)
print("Dataset:", RUTA_DATASET.resolve())


## Preparaci?n de archivos

Las im?genes TIFF se leen mediante PIL para mantener compatibilidad con el dataset original.


In [ ]:
def obtener_archivos(ruta):

    clases = sorted([
        carpeta.name
        for carpeta in ruta.iterdir()
        if carpeta.is_dir()
    ])

    mapa_clases = {
        nombre: indice
        for indice, nombre in enumerate(clases)
    }

    rutas = []
    etiquetas = []

    for clase in clases:

        carpeta = ruta / clase

        archivos = [
            archivo
            for archivo in carpeta.rglob("*")
            if archivo.is_file()
            and archivo.suffix.lower() in EXTENSIONES
        ]

        for archivo in archivos:

            rutas.append(str(archivo))
            etiquetas.append(mapa_clases[clase])

    return rutas, etiquetas, clases


rutas_train, etiquetas_train, clases = obtener_archivos(RUTA_TRAIN)
rutas_val, etiquetas_val, _ = obtener_archivos(RUTA_VALIDACION)
rutas_test, etiquetas_test, _ = obtener_archivos(RUTA_TEST)

print("Clases:", clases)
print("Train:", len(rutas_train))
print("Validaci?n:", len(rutas_val))
print("Test:", len(rutas_test))


In [ ]:
def leer_imagen(ruta, etiqueta):

    ruta = ruta.numpy().decode("utf-8")

    with Image.open(ruta) as imagen:

        imagen = imagen.convert("RGB")
        imagen = imagen.resize((ANCHO, ALTO))

        matriz = np.asarray(
            imagen,
            dtype=np.float32
        )

    return matriz, np.int32(etiqueta)


def procesar_imagen(ruta, etiqueta):

    imagen, etiqueta = tf.py_function(
        func=leer_imagen,
        inp=[ruta, etiqueta],
        Tout=[tf.float32, tf.int32]
    )

    imagen.set_shape((ALTO, ANCHO, 3))
    etiqueta.set_shape(())

    return imagen, etiqueta


def crear_dataset(rutas, etiquetas, entrenamiento=False):

    dataset = tf.data.Dataset.from_tensor_slices(
        (rutas, etiquetas)
    )

    if entrenamiento:

        dataset = dataset.shuffle(
            len(rutas),
            seed=SEMILLA
        )

    dataset = dataset.map(
        procesar_imagen,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(TAMANO_LOTE)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


dataset_train = crear_dataset(
    rutas_train,
    etiquetas_train,
    entrenamiento=True
)

dataset_val = crear_dataset(
    rutas_val,
    etiquetas_val
)

dataset_test = crear_dataset(
    rutas_test,
    etiquetas_test
)


## Modelo baseline

Se utiliza MobileNetV3Small con pesos preentrenados de ImageNet.

Inicialmente el extractor de caracter?sticas permanece congelado.


In [ ]:
aumento_datos = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomContrast(0.08)
])

modelo_base = tf.keras.applications.MobileNetV3Small(
    input_shape=(ALTO, ANCHO, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=True
)

modelo_base.trainable = False

entrada = tf.keras.Input(
    shape=(ALTO, ANCHO, 3)
)

x = aumento_datos(entrada)

x = modelo_base(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.35)(x)

salida = tf.keras.layers.Dense(
    len(clases),
    activation="softmax"
)(x)

modelo = tf.keras.Model(
    entrada,
    salida
)

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo.summary()


## Pesos de clase

Los pesos se calculan utilizando solamente las im?genes de entrenamiento.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

clases_numericas = np.unique(etiquetas_train)

pesos = compute_class_weight(
    class_weight="balanced",
    classes=clases_numericas,
    y=np.array(etiquetas_train)
)

pesos_clases = {
    int(clase): float(peso)
    for clase, peso in zip(
        clases_numericas,
        pesos
    )
}

print("Pesos:", pesos_clases)


## Entrenamiento

Configuraci?n inicial:

- Adam.
- Learning rate: 0.001.
- Batch: 8.
- Hasta 35 ?pocas.
- Early stopping con paciencia 7.


In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "mejor_baseline.keras",
        monitor="val_loss",
        save_best_only=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3
    )
]


historial = modelo.fit(
    dataset_train,
    validation_data=dataset_val,
    epochs=35,
    class_weight=pesos_clases,
    callbacks=callbacks
)


## Curvas de entrenamiento


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    historial.history["loss"],
    label="Train"
)

plt.plot(
    historial.history["val_loss"],
    label="Validaci?n"
)

plt.xlabel("?poca")
plt.ylabel("Loss")
plt.title("P?rdida de entrenamiento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    historial.history["accuracy"],
    label="Train"
)

plt.plot(
    historial.history["val_accuracy"],
    label="Validaci?n"
)

plt.xlabel("?poca")
plt.ylabel("Accuracy")
plt.title("Accuracy de entrenamiento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Evaluaci?n final

La evaluaci?n se realiza exclusivamente con el conjunto de prueba.


In [ ]:
probabilidades = modelo.predict(
    dataset_test
)

predicciones = np.argmax(
    probabilidades,
    axis=1
)

verdaderos = np.array(
    etiquetas_test
)

print(
    classification_report(
        verdaderos,
        predicciones,
        target_names=clases,
        digits=4
    )
)

accuracy_balanceada = balanced_accuracy_score(
    verdaderos,
    predicciones
)

print(
    "Balanced Accuracy:",
    round(accuracy_balanceada, 4)
)

matriz = confusion_matrix(
    verdaderos,
    predicciones
)

ConfusionMatrixDisplay(
    matriz,
    display_labels=clases
).plot(
    cmap="Blues"
)

plt.title("Matriz de confusi?n - Baseline")
plt.show()
